# agent2society quickstart

A short notebook tour of the package. We will:

1. Build a 3-agent society from plain Python objects.
2. Run a task with a bare string (the simplest possible call).
3. Run a task with a `Handoff` envelope and pull the routing explanation.
4. Surface an agent's `SelfAssessment` caveats in the explanation.
5. Wire detection-only governance hooks (low confidence, human review, conflict).
6. Chain two handoffs so the second decision carries the first as `prior`.
7. Print the per-task ledger via `society.report()`.

Everything here runs locally with no LLM, no network, no API keys.

## 0. Install

```bash
pip install agent2society
```

If you are running this from inside the repo:

```bash
pip install -e .
```

In [ ]:
import agent2society
from agent2society import Society, Handoff

print("agent2society version:", agent2society.__version__)

## 1. Build a society from plain Python objects

An agent in agent2society is anything that exposes a `name`, `description`, and `skills` list, plus a callable entry point (`run`, `invoke`, `kickoff`, or `__call__`). The adapter layer wraps these into an A2A card automatically -- no JSON file required for local agents.

In [ ]:
class Researcher:
    name = "research-agent"
    description = "Searches the web and summarises sources."
    skills = [
        {
            "id": "web_research",
            "name": "Web Research",
            "description": "Search the web, gather sources, summarise findings.",
            "tags": ["research", "web", "search", "sources"],
        }
    ]

    def __call__(self, task):
        return f"[research] {task}"


class Writer:
    name = "writer-agent"
    description = "Drafts executive memos from notes."
    skills = [
        {
            "id": "exec_memo",
            "name": "Executive Memo",
            "description": "Draft an executive memo from notes or findings.",
            "tags": ["writing", "memo", "exec", "draft", "business"],
        }
    ]

    def run(self, task):
        return f"[memo] {task}"


class Coder:
    name = "coder-agent"
    description = "Writes Python code."
    skills = [
        {
            "id": "python_code",
            "name": "Write Python",
            "description": "Write Python functions, refactor, debug.",
            "tags": ["code", "python", "programming"],
        }
    ]

    def invoke(self, task):
        return f"```python\n# {task}\n```"


society = Society()
society.add(Researcher())
society.add(Writer())
society.add(Coder())

society.agents()

## 2. Run a task with a bare string

Bare strings still work. agent2society wraps them in a `Handoff` internally so every dispatch has a stable id to key explanations against.

In [ ]:
print(society.run("Research Q3 customer churn drivers from the web"))
print(society.run("Write a Python function to dedupe a list of dicts"))

## 3. Run a task with a Handoff and pull the explanation

A `Handoff` carries the *meaning* around the task: the business intent, assumptions, prior decisions, and a confidence threshold. The handoff `id` is what `society.explain(id)` keys against later.

In [ ]:
h = Handoff(
    task="Draft an executive memo on Q3 customer churn",
    intent="prep the board pack",
    assumptions=["churn data is final"],
    confidence_required=0.5,
)

result = society.run(h)
print("result:", result)
print()
print(society.explain(h.id).render())

Notice what the explanation contains:

- **chose**: the picked `(agent, skill)` and the score it cleared.
- **why**: a deterministic sentence built from the features that actually fired -- no LLM produced this.
- **features**: the raw numbers (score, semantic similarity, tag overlap, the threshold the router compared against).
- **alternatives**: every other candidate and why it lost (`REJECTED` lines carry a `reason`).
- **agent self-caveats**: if the chosen agent's card declares limits, they appear here. Our `Writer` declared none, so we get the default placeholder. Next cell fixes that.

## 4. Declare agent limits via SelfAssessment

An A2A card can carry a `selfAssessment` block. agent2society surfaces those limits on every routing explanation that picks the agent.

Below we wire the writer through a full A2A card instead of an adapter so we can attach the block, and register a tiny local handler so dispatch still works in-process.

In [ ]:
WRITER_CARD = {
    "name": "writer-agent",
    "url": "local://writer",
    "description": "Drafts executive memos from notes.",
    "skills": [
        {
            "id": "exec_memo",
            "name": "Executive Memo",
            "description": "Draft an executive memo from notes or findings.",
            "tags": ["writing", "memo", "exec", "draft", "business"],
        }
    ],
    "selfAssessment": {
        "confidenceModel": "tfidf_score",
        "knownLimitations": ["English only", "max ~400 words per memo"],
        "outOfScope": ["legal opinion", "binding financial guidance"],
        "escalateWhen": ["any quantitative claim cited without source"],
    },
}


def writer_handler(_url, payload):
    return {
        "jsonrpc": "2.0",
        "id": payload.get("id", "0"),
        "result": {"parts": [{"kind": "text", "text": "MEMO: churn driven by pricing"}]},
    }


society2 = Society()
society2.add(Researcher())
card = society2.add(WRITER_CARD)
society2._local.register(card.url, writer_handler)

h2 = Handoff(task="Draft an executive memo on churn")
society2.run(h2)
print(society2.explain(h2.id).render())

The `agent self-caveats` block now lists everything the writer card declared. Callers (and downstream agents in a chain) see the same limits the agent claims for itself.

## 5. Governance hooks

Four hooks ship on the society. They fire as side effects of `society.run()`. **They cannot block, retry, or modify a dispatch.** Handler exceptions are swallowed. The package never silently auto-corrects.

- `on_low_confidence(handler, threshold=...)` -- fires when a decision's confidence is below the handoff's `confidence_required` (or this society-wide threshold).
- `on_human_review(handler)` -- fires when a `Handoff.human_review_when(result_text)` predicate returns True.
- `on_conflict(handler)` -- fires when the same task was routed to different `(agent, skill)` pairs across handoffs in the window.
- `on_capability_drift(handler)` -- fires when one agent is selected across an unusually broad spread of skills.

In [ ]:
low_conf_log = []
review_log = []

society2.on_low_confidence(
    lambda exp: low_conf_log.append((exp.task, round(exp.confidence, 3))),
    threshold=0.8,  # tight on purpose so we see it fire
)
society2.on_human_review(lambda exp, text: review_log.append(text))

h_review = Handoff(
    task="Draft an executive memo on Q3 churn",
    human_review_when=lambda text: "MEMO" in text,
)
society2.run(h_review)

print("low-confidence triggers:", low_conf_log)
print("human-review triggers:  ", review_log)

## 6. Handoff chains -- carry the upstream decision forward

`handoff.extend(...)` produces the next handoff in the chain. The new handoff carries a `DecisionRecord` describing what the upstream agent did, so the downstream routing explanation includes the chain in its `prior` field.

In [ ]:
h_research = Handoff(
    task="Research Q3 customer churn drivers",
    intent="prep the Q3 board pack",
    assumptions=["churn data through end-of-quarter is final"],
)
society2.run(h_research)

h_memo = h_research.extend(
    agent="research-agent",
    skill="web_research",
    summary="found 3 churn drivers: pricing, onboarding, support latency",
    confidence=0.62,
    next_task="Draft an executive memo on Q3 churn drivers",
)
society2.run(h_memo)

print(society2.explain(h_memo.id).render())

The downstream explanation now shows the chain:

```
chain  : 1 prior step(s)
  1. research-agent :: web_research -> found 3 churn drivers...
```

That's how meaning survives the handoff: the next agent (and any human auditing the trail) sees why this work exists.

## 7. Conformance guardrail

Before dispatch, agent2society checks (a) the agent's card actually declares this skill and (b) the task is inside the agent's allow/deny boundary. If either fails, the dispatch is blocked.

In `strict=False` mode the run returns an empty string and the explanation carries a `blocked_reason`:

In [ ]:
society3 = Society(strict=False)
society3.add(Writer())
society3.boundary("writer-agent", deny=["financial-data"])

h_blocked = Handoff(task="Draft an exec memo including financial-data figures")
out = society3.run(h_blocked)
exp = society3.explain(h_blocked.id)

print("output (empty):", repr(out))
print("blocked_reason:", exp.blocked_reason)

In default `strict=True` mode the same call raises `agent2society.ConformanceViolation` instead -- the agent never silently accepts work outside its declared boundary.

## 8. Telemetry: `society.report()`

Per-task ledger: chosen agent + skill, score, fallbacks, conformance violations, a token estimate, and the `society.explain(handoff_id)` hint so you can pull the full rationale on any line.

In [ ]:
society2.report()

## What next

- Swap the default TF-IDF scorer for real embeddings: `Society(embed_fn=lambda texts: model.encode(texts).tolist())`.
- Drop a CrewAI `Crew`, a compiled LangGraph, or an AutoGen `ConversableAgent` straight into `society.add(...)` -- the adapter layer handles them.
- Wire `society.add("https://your-agent.example.com/.well-known/agent-card.json")` to route to remote A2A agents.
- Inspect the supervisor head-to-head benchmark: `python benchmarks/run.py`.

See [`examples/transparent_mesh.py`](transparent_mesh.py) for a single-file walkthrough of the same surface as a runnable script.